# Dr Plant — Lemon baseline training (clean Colab v1)

Run this notebook from top to bottom in Google Colab. Select a GPU runtime if available. It downloads only the **raw** `Project-AgML/lemon_leaf_disease_classification` subset (expected 1,354 rows / 9 classes; source card states CC BY 4.0, upstream DOI `10.17632/44nrn4593f.1`). It audits images, quarantines exact duplicate bytes with conflicting labels, creates a reproducible exploratory split, trains MobileNetV3-Small, exports ONNX, validates the ONNX graph and inference, and saves all outputs to Google Drive.

**Important:** this dataset does not provide verified site/plant/session groups or independent expert label review. The run is therefore an engineering/exploratory baseline only—not validated accuracy for Syrian orchards and not production diagnosis. Do not remove the safeguards or present these metrics as independent field accuracy.

In [ ]:
# 1. Mount Drive and prepare a clean workspace. Safe to rerun; raw cache and previous runs are preserved.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, os, time

REPO = Path('/content/dr-plant')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','training/lemon-clean-colab-v1','https://github.com/jonbander/dr-plant.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
WORK = Path('/content/drplant-lemon-v1')
DRIVE_ROOT = Path('/content/drive/MyDrive/DrPlant')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime('%Y%m%d-%H%M%S')
RUN_DIR = DRIVE_ROOT / f'lemon-clean-v1-{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Repository:', REPO)
print('Run output:', RUN_DIR)
import torch, torchvision
print('PyTorch:', torch.__version__, '| TorchVision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
# Static syntax check: validate every notebook code cell before any dataset work starts.
import ast
notebook_path = REPO / 'notebooks' / 'DrPlant_Lemon_Clean_Training_v1.ipynb'
notebook_json = json.loads(notebook_path.read_text(encoding='utf-8'))
for cell_number, cell in enumerate(notebook_json['cells']):
    if cell.get('cell_type') == 'code':
        source = ''.join(cell.get('source', []))
        ast.parse(source, filename=f'notebook-cell-{cell_number}')
print('Notebook Python syntax check: PASS')

In [ ]:
# 2. Install only the data/export dependencies. Colab's preinstalled torch/torchvision are not replaced.
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=2.18','Pillow>=10','scikit-learn>=1.4','onnx>=1.15','onnxruntime>=1.17'], check=True)
from datasets import load_dataset
from PIL import Image
import numpy as np
print('Dependencies imported successfully.')

In [ ]:
# 3. Download/cache the exact RAW subset and convert it to class folders.
HF_DATASET = 'Project-AgML/lemon_leaf_disease_classification'
DATASET_ID = '44nrn4593f'
IMAGE_ROOT = DRIVE_ROOT / 'lemon-agml-raw-images'
READY = IMAGE_ROOT / '.drplant-dataset-ready.json'
HF_CACHE = DRIVE_ROOT / 'huggingface-cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}

def count_images(root):
    return sum(1 for p in root.glob('*/*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if root.is_dir() else 0

if READY.is_file() and count_images(IMAGE_ROOT) >= 1000:
    dataset_info = json.loads(READY.read_text(encoding='utf-8'))
    print('Reusing cached source conversion:', dataset_info)
else:
    ds = load_dataset(HF_DATASET, 'raw', split='train', cache_dir=str(HF_CACHE))
    label_feature = ds.features.get('label')
    if not hasattr(label_feature, 'names') or not label_feature.names:
        raise RuntimeError('Dataset schema changed: expected ClassLabel feature named label. Stopped without guessing labels.')
    class_names = list(label_feature.names)
    if len(ds) != 1354 or len(class_names) != 9:
        raise RuntimeError(f'Unexpected dataset revision: rows={len(ds)}, classes={len(class_names)}. Review source/version before proceeding.')
    IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    counts = {name: 0 for name in class_names}
    from io import BytesIO
    for i, row in enumerate(ds):
        label_idx = int(row['label'])
        if not 0 <= label_idx < len(class_names): raise RuntimeError(f'Invalid label index at row {i}: {label_idx}')
        image_obj = row['image']
        if isinstance(image_obj, dict):
            if image_obj.get('bytes'): image_obj = Image.open(BytesIO(image_obj['bytes']))
            elif image_obj.get('path'): image_obj = Image.open(image_obj['path'])
            else: raise RuntimeError(f'Missing image bytes/path at row {i}')
        if not isinstance(image_obj, Image.Image): raise RuntimeError(f'Unexpected image type at row {i}: {type(image_obj)}')
        safe_name = ''.join(ch.lower() if ch.isalnum() or ch in '-_' else '_' for ch in class_names[label_idx]).strip('_')
        target = IMAGE_ROOT / safe_name
        target.mkdir(parents=True, exist_ok=True)
        image_obj.convert('RGB').save(target / f'{counts[class_names[label_idx]]:06d}.jpg', quality=95)
        counts[class_names[label_idx]] += 1
        if (i+1) % 200 == 0 or i+1 == len(ds): print(f'Converted {i+1}/{len(ds)}')
    if sum(counts.values()) != len(ds) or any(v == 0 for v in counts.values()): raise RuntimeError(f'Conversion count/class check failed: {counts}')
    dataset_info = {'dataset':HF_DATASET,'upstream_doi':'10.17632/44nrn4593f.1','subset':'raw','rows':len(ds),'class_names':class_names,'class_counts':counts}
    READY.write_text(json.dumps(dataset_info,ensure_ascii=False,indent=2),encoding='utf-8')

actual_count = count_images(IMAGE_ROOT)
if actual_count != 1354: raise RuntimeError(f'Expected 1,354 image files after conversion/cache; found {actual_count}.')
print('Dataset image count:', actual_count)
print('Classes:', dataset_info.get('class_names', dataset_info.get('class_counts',{}).keys()))

In [ ]:
# 4. Audit every image: decode checks, SHA-256 exact duplicates, class counts. This does not certify biological labels.
AUDIT = WORK / 'audit'
if AUDIT.exists(): import shutil; shutil.rmtree(AUDIT)
cmd = [sys.executable,str(REPO/'scripts/audit_image_classification_dataset.py'),str(IMAGE_ROOT),'--dataset-id',DATASET_ID,'--version','1','--license','CC BY 4.0 (dataset card claim; verify source terms before redistribution)','--out',str(AUDIT),'--verify-images']
result = subprocess.run(cmd,text=True,capture_output=True)
print(result.stdout)
if result.returncode != 0: print(result.stderr); raise RuntimeError('Dataset audit failed; training stopped.')
audit_report = json.loads((AUDIT/'archive-audit-report.json').read_text(encoding='utf-8'))
if audit_report['image_count'] != 1354: raise RuntimeError(f"Audit count mismatch: {audit_report['image_count']}")
if audit_report['unreadable_images']: raise RuntimeError(f"Unreadable images found: {len(audit_report['unreadable_images'])}; inspect audit report before training.")
if len(audit_report['class_image_counts']) != 9: raise RuntimeError('Audit did not find all nine expected classes.')
print('Exact duplicate groups:', audit_report['exact_duplicate_groups'])
print('Class counts:', audit_report['class_image_counts'])

In [ ]:
# 5. Create a deterministic exploratory split. Conflicting-label identical images are quarantined.
# This split is intentionally NOT claimed independent by orchard/site/plant/session.
SPLITS = WORK / 'exploratory-splits'
if SPLITS.exists(): import shutil; shutil.rmtree(SPLITS)
cmd = [sys.executable,str(REPO/'scripts/prepare_exploratory_splits.py'),str(AUDIT/'image-manifest.csv'),'--out',str(SPLITS),'--seed','42']
result = subprocess.run(cmd,text=True,capture_output=True)
print(result.stdout)
if result.returncode != 0: print(result.stderr); raise RuntimeError('Split failed or data was blocked after duplicate quarantine. Training was NOT started.')
split_report = json.loads((SPLITS/'exploratory-split-report.json').read_text(encoding='utf-8'))
if split_report['images_in_splits'] < 1000: raise RuntimeError('Too few unambiguous images remain after quarantine; do not weaken this gate.')
split_rows = list(__import__('csv').DictReader((SPLITS/'exploratory-splits.csv').open(encoding='utf-8')))
all_classes = {r['class_label'] for r in split_rows}
for part in ('train','validation','test'):
    if {r['class_label'] for r in split_rows if r['split']==part} != all_classes: raise RuntimeError(f'Class coverage missing in {part} split.')
print('Images per split:', split_report['images_per_split'])
print('Conflicting-label images quarantined:', split_report['quarantined_conflicting_label_images'])

In [ ]:
# 6. Train MobileNetV3-Small. The script selects the best validation checkpoint and evaluates test once.
MODEL_OUT = WORK / 'model-output'
if MODEL_OUT.exists(): import shutil; shutil.rmtree(MODEL_OUT)
cmd = [sys.executable,str(REPO/'scripts/train_image_classifier.py'),str(IMAGE_ROOT),str(SPLITS/'exploratory-splits.csv'),'--out',str(MODEL_OUT),'--epochs','12','--batch-size','32','--seed','42','--pretrained','--exploratory-non-independent']
result = subprocess.run(cmd,text=True,capture_output=True)
print(result.stdout)
if result.returncode != 0: print(result.stderr); raise RuntimeError('Training failed. No metrics are being reported as successful.')
metrics = json.loads((MODEL_OUT/'metrics.json').read_text(encoding='utf-8'))
if metrics.get('status') != 'exploratory_non_independent_baseline': raise RuntimeError('Unexpected training status.')
if not (MODEL_OUT/'best-model.pt').is_file(): raise RuntimeError('Checkpoint file missing after training.')
print('Best validation accuracy:', metrics['best_validation_accuracy'])
print('Exploratory test accuracy:', metrics['test_accuracy'])
print('Exploratory test macro-F1:', metrics['test_macro_f1'])
print('Per-class report:', json.dumps(metrics['test_classification_report'],ensure_ascii=False,indent=2))

In [ ]:
# 7. Export ONNX and validate both the graph and a real inference using ONNX Runtime.
import torch, onnx, onnxruntime as ort
from torch import nn
from torchvision import models
classes = metrics['classes']
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features,len(classes))
checkpoint = torch.load(MODEL_OUT/'best-model.pt',map_location='cpu',weights_only=False)
model.load_state_dict(checkpoint['state_dict'],strict=True)
model.eval()
onnx_path = MODEL_OUT/'lemon-mobilenetv3-small.onnx'
dummy = torch.zeros(1,3,224,224,dtype=torch.float32)
torch.onnx.export(model,dummy,str(onnx_path),input_names=['images'],output_names=['logits'],dynamic_axes={'images':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)
graph = onnx.load(str(onnx_path)); onnx.checker.check_model(graph)
session = ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
with torch.no_grad(): expected = model(dummy).numpy()
actual = session.run(['logits'],{'images':dummy.numpy()})[0]
max_abs_error = float(np.max(np.abs(expected-actual)))
if actual.shape != (1,len(classes)) or not np.isfinite(actual).all() or max_abs_error > 1e-3:
    raise RuntimeError(f'ONNX inference validation failed: shape={actual.shape}, max_abs_error={max_abs_error}')
labels = {'architecture':'MobileNetV3-Small','input_name':'images','output_name':'logits','input_shape':[1,3,224,224],'resize':[224,224],'color_mode':'RGB','normalization':{'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225]},'classes':classes,'evaluation_scope':'exploratory image-level split; not independent by site/plant/session'}
(MODEL_OUT/'labels.json').write_text(json.dumps(labels,ensure_ascii=False,indent=2)+'\\n',encoding='utf-8')
print('ONNX file:', onnx_path, '| bytes:', onnx_path.stat().st_size)
print('ONNX checker: PASS | ONNX Runtime inference: PASS | max abs error:', max_abs_error)

In [ ]:
# 8. Save complete, timestamped results to Google Drive. Previous runs are not overwritten.
import shutil
for source in (AUDIT,SPLITS,MODEL_OUT):
    target = RUN_DIR / source.name
    if target.exists(): shutil.rmtree(target)
    shutil.copytree(source,target)
shutil.copy2(READY,RUN_DIR/'dataset-source-info.json')
summary = {'run_id':RUN_ID,'dataset':dataset_info,'audit':audit_report,'split':split_report,'metrics':metrics,'onnx':{'path':'model-output/lemon-mobilenetv3-small.onnx','validated':True,'max_abs_error':max_abs_error},'limitations':['Exploratory image-level holdout; not independent by site/plant/session.','Dataset labels and pathogen identities have not been independently expert-confirmed here.','Not validated on Syrian coastal field images; not production-ready.']}
(RUN_DIR/'run-summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\\n',encoding='utf-8')
print('All outputs saved to:',RUN_DIR)
print('Final status: TRAINED + ONNX VALIDATED (EXPLORATORY ONLY)')